In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [3]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="ai4bharat/Rasa", 
    repo_type="dataset", local_dir="./Rasa", allow_patterns="data/*.parquet")

Fetching 488 files: 100%|██████████| 488/488 [07:54<00:00,  1.03it/s]


'/home/ubuntu/Rasa'

In [4]:
files = glob('Rasa/*/*.parquet')
len(files)

488

In [5]:
df = pd.read_parquet(files[0])
df

,filename,text,language,gender,style,duration,wav_path,audio
0,TEL_F_WIKI_02450,"దానితో పాటు, పంజాబుపై ఆఫ్ఘన్ల నియంత్రణను కూడా ...",Telugu,Female,WIKI,16.184,/projects/data/ttsteam/datasets/rasa_hf/data/T...,{'bytes': b'RIFF$\xb5\x17\x00WAVEfmt \x10\x00\...
1,TEL_F_BOOK_00822,"""అయితే ఇప్పుడు ఎందుకు చూపించినట్లు తమ మనసు తెల...",Telugu,Female,BOOK,5.107,/projects/data/ttsteam/datasets/rasa_hf/data/T...,{'bytes': b'RIFFD{\x07\x00WAVEfmt \x10\x00\x00...
2,TEL_F_SURPRISE_00065,ఒకే డెలివరీకి నా దగ్గర రెండుసార్లు వసూలు చేశార...,Telugu,Female,SURPRISE,6.987,/projects/data/ttsteam/datasets/rasa_hf/data/T...,{'bytes': b'RIFFD<\n\x00WAVEfmt \x10\x00\x00\x...
3,TEL_F_CONV_02286,అక్కడ సందర్శించటానికి కొన్ని ప్రసిద్ధ గురుద్వా...,Telugu,Female,CONV,5.826,/projects/data/ttsteam/datasets/rasa_hf/data/T...,{'bytes': b'RIFF\xe4\x88\x08\x00WAVEfmt \x10\x...
4,TEL_F_BB_00097,"గ్రుడ్లు, గుడ్లు, చాకులు చెంచాలు, కత్తులూ, కటా...",Telugu,Female,BB,7.581,/projects/data/ttsteam/datasets/rasa_hf/data/T...,{'bytes': b'RIFF\x04\x1b\x0b\x00WAVEfmt \x10\x...
...,...,...,...,...,...,...,...,...
847,TEL_F_CONV_02920,"ఆహా, అర్ధమైంది.",Telugu,Female,CONV,1.816,/projects/data/ttsteam/datasets/rasa_hf/data/T...,{'bytes': b'RIFF$\xa9\x02\x00WAVEfmt \x10\x00\...
848,TEL_F_WIKI_03126,ఈ ప్రాంతంలోని వాయువ్య భాగాలు మోన్యుల్ యొక్క మన...,Telugu,Female,WIKI,11.310,/projects/data/ttsteam/datasets/rasa_hf/data/T...,{'bytes': b'RIFFd\x91\x10\x00WAVEfmt \x10\x00\...
849,TEL_F_BOOK_00556,శరీరమంతా చల్లగా మంచుముద్దయి పోతోంది.,Telugu,Female,BOOK,3.116,/projects/data/ttsteam/datasets/rasa_hf/data/T...,{'bytes': b'RIFF\xa4\x90\x04\x00WAVEfmt \x10\x...
850,TEL_F_WIKI_01703,ఒక అనామక శత్రువు మనతో ఉన్నారు.,Telugu,Female,WIKI,3.515,/projects/data/ttsteam/datasets/rasa_hf/data/T...,{'bytes': b'RIFFD&\x05\x00WAVEfmt \x10\x00\x00...


In [6]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [7]:
data = multiprocessing(files, loop, cores = 20)

100%|██████████| 8/8 [10:40<00:00, 80.01s/it]


In [8]:
len(data)

412347

In [9]:
data[0]

{'audio_filename': 'Rasa_audio/Rasa-data-train-00413-of-00436_0.mp3',
 'text': 'దానితో పాటు, పంజాబుపై ఆఫ్ఘన్ల నియంత్రణను కూడా బలహీనపరిచాడు, సామ్రాజ్య రాజధాని ఢిల్లీపై వారి పదేపదే దండయాత్రలను అడ్డగించాడు, రాజపుత్రులను, రోహిల్లాలను లొంగదీసుకున్నాడు మరియు ఔధ్ రాజ్యాన్ని ఓడించాడు.',
 'speaker': 'Rasa_audio'}

In [10]:
with open('Rasa.json', 'w') as fopen:
    json.dump(data, fopen)

In [11]:
audio_files = [d['audio_filename'] for d in data]

with open('Rasa-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [3]:
# !zip -rq Rasa_audio_neucodec.zip Rasa_audio_neucodec

In [4]:
# !hf upload malaysia-ai/Multilingual-TTS Rasa_audio_neucodec.zip --repo-type=dataset

In [8]:
# !zip -rq Rasa_audio.zip Rasa_audio

In [9]:
# !hf upload malaysia-ai/Multilingual-TTS Rasa_audio.zip --repo-type=dataset

In [11]:
import json
from tqdm import tqdm

with open('Rasa.json') as fopen:
    rows = json.load(fopen)

mapping = {}
for i in tqdm(range(len(rows))):
    mapping[rows[i]['audio_filename']] = i
len(mapping)

100%|██████████| 412347/412347 [00:00<00:00, 2700032.74it/s]


412347

In [12]:
import faiss
import os
import numpy as np
from tqdm import tqdm

data = {}
d = 192
index = faiss.IndexFlatL2(d)

centroids = []

def assign(x, threshold=0.1):
    if len(centroids) == 0:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return 0
    
    D, I = index.search(np.array([x], dtype=np.float32), 1)
    if D[0][0] > threshold:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return len(centroids)-1
    else:
        return I[0][0]
        
for i in tqdm(range(len(rows))):
    index_ = mapping[rows[i]['audio_filename']]
    v_f = f'Rasa_embedding/{index_}.npy'
    if not os.path.exists(v_f):
        continue
    try:
        v = np.load(v_f)
        data[rows[i]['audio_filename']] = assign(v)
    except Exception as e:
        pass

100%|██████████| 412347/412347 [01:55<00:00, 3554.75it/s]


In [13]:
for i in range(len(rows)):
    s = data[rows[i]['audio_filename']]
    rows[i]['speaker'] = rows[i]['speaker'] + f'_{s}'

In [14]:
from datasets import Dataset

dataset = Dataset.from_list(rows)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'Rasa_audio/Rasa-data-train-00413-of-00436_0.mp3',
 'text': 'దానితో పాటు, పంజాబుపై ఆఫ్ఘన్ల నియంత్రణను కూడా బలహీనపరిచాడు, సామ్రాజ్య రాజధాని ఢిల్లీపై వారి పదేపదే దండయాత్రలను అడ్డగించాడు, రాజపుత్రులను, రోహిల్లాలను లొంగదీసుకున్నాడు మరియు ఔధ్ రాజ్యాన్ని ఓడించాడు.',
 'speaker': 'Rasa_audio_0'}

In [15]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'Rasa')

Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00,  7.81ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1): 100%|█████████▉| 40.5MB / 40.7MB, 6.14MB/s  
Processing Files (1 / 1): 100%|██████████| 40.7MB / 40.7MB, 5.65MB/s  
Processing Files (1 / 1): 100%|██████████| 40.7MB / 40.7MB, 5.09MB/s  
New Data Upload: 100%|██████████| 40.7MB / 40.7MB, 5.09MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:08<00:00,  8.89s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/5142120b549d2a1906d7045a799e0dbd24b5a3dc', commit_message='Upload dataset', commit_description='', oid='5142120b549d2a1906d7045a799e0dbd24b5a3dc', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)